In [3]:
import numpy as np

ker = np.loadtxt("conv1_kernel_000.txt", dtype=np.int16)  # shape (9,)

print("constant K0 : kernel3x3_t := (")
for i, v in enumerate(ker):
    sep = "," if i < 8 else ""
    print(f"  {i} => to_signed({int(v)}, 16){sep}")
print(");")

constant K0 : kernel3x3_t := (
  0 => to_signed(-59, 16),
  1 => to_signed(46, 16),
  2 => to_signed(-46, 16),
  3 => to_signed(-30, 16),
  4 => to_signed(35, 16),
  5 => to_signed(-3, 16),
  6 => to_signed(44, 16),
  7 => to_signed(-45, 16),
  8 => to_signed(19, 16)
);


In [ ]:
import numpy as np
from pathlib import Path

# === Paramètres à adapter si besoin ===
N_FILTERS   = 32                 # taille de la banque (0..31)
WIDTH       = 16                 # largeur des poids en VHDL (signed(WIDTH-1 downto 0))
CONST_NAME  = "K_BANK32_L0"      # nom de la constante VHDL
TYPE_NAME   = "kernel3x3_bank32_t"
k = 0                            # indice du groupe de kernels à charger (0..31)        

# Dossier et pattern des fichiers de kernels
FILE_PATTERN = "conv1_kernel_{:03d}.txt"  # conv1_kernel_000.txt, 001, ...

def load_kernel(idx: int) -> np.ndarray:
    """Charge un kernel 3x3 depuis un fichier texte (9 int16)."""
    fname = FILE_PATTERN.format(idx)
    ker = np.loadtxt(fname, dtype=np.int16)
    if ker.shape[0] != 9:
        raise ValueError(f"Kernel {fname} ne contient pas 9 valeurs (shape = {ker.shape})")
    return ker

print(f"constant {CONST_NAME} : {TYPE_NAME} := (")
for f in range(k*N_FILTERS,(k+1)*N_FILTERS):
    ker = load_kernel(f)
    print(f"  {f%32} => (")
    for i, v in enumerate(ker):
        sep = "," if i < 8 else ""
        print(f"    {i} => to_signed({int(v)}, {WIDTH}){sep}")
    # virgule après le filtre sauf pour le dernier
    end_sep = "," if f < N_FILTERS - 1 else ""
    print(f"  ){end_sep}")
print(");")


constant K_BANK32_L0 : kernel3x3_bank32_t := (
  0 => (
    0 => to_signed(94, 16),
    1 => to_signed(0, 16),
    2 => to_signed(55, 16),
    3 => to_signed(-35, 16),
    4 => to_signed(26, 16),
    5 => to_signed(21, 16),
    6 => to_signed(-79, 16),
    7 => to_signed(39, 16),
    8 => to_signed(-24, 16)
  )
  1 => (
    0 => to_signed(30, 16),
    1 => to_signed(46, 16),
    2 => to_signed(-66, 16),
    3 => to_signed(18, 16),
    4 => to_signed(16, 16),
    5 => to_signed(-57, 16),
    6 => to_signed(-27, 16),
    7 => to_signed(9, 16),
    8 => to_signed(-1, 16)
  )
  2 => (
    0 => to_signed(72, 16),
    1 => to_signed(63, 16),
    2 => to_signed(-62, 16),
    3 => to_signed(8, 16),
    4 => to_signed(-6, 16),
    5 => to_signed(14, 16),
    6 => to_signed(-14, 16),
    7 => to_signed(-59, 16),
    8 => to_signed(5, 16)
  )
  3 => (
    0 => to_signed(-38, 16),
    1 => to_signed(69, 16),
    2 => to_signed(39, 16),
    3 => to_signed(-20, 16),
    4 => to_signed(21, 16),
    5

In [ ]:
IMG_W, IMG_H = 12, 98
SCALE_MFCC = 2**15

# À adapter : recharger ton exemple X_val[0] depuis disque si nécessaire
x = X_val[0]          # (98,12,1) ou (98,12)
x2d = x[..., 0] if x.ndim == 3 else x

img_q = np.clip(np.round(x2d * SCALE_MFCC), -32768, 32767).astype(np.int16)

# Sauvegarde pour le TB
np.savetxt("img_q_int16.txt", img_q.reshape(-1), fmt="%d")
